# Non-Overlapping Patch Composition By Embedding Cluster

This notebook explores the validation-patch workflow used by `scripts/export_patch_composition_tables.py`.

1. Train a depth-2 `GINCurvature` model.
2. Cluster all validation nodes using their final local node embedding.
3. Sample strictly node-disjoint radius-2 patches within each organoid.
4. Assign each patch the embedding cluster of its center cell.
5. Summarize patch marker composition by cluster.
6. Within each cluster, retain the 50% of patches with the smallest center-cell absolute prediction error and repeat the marker analysis.

Marker statistics treat each sampled patch as one observation and use the fraction of marker-positive cells within the patch.


**Setup**

In [ ]:
import copy
import json
import pickle
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "training_data"
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


**Settings**

In [ ]:
# Data and filtering. Defaults match export_patch_composition_tables.py.
EXPERIMENT_GROUP = "patch_composition_sampling_analysis"
DATASET_NAME = "mean_curvature_smooth"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_ANALYSIS = 0
EXCLUDE_TIMEPOINT = "day4"  # Set to None to retain every timepoint.
FILTER_BLACKLISTED_ORGANOIDS = False
APPLY_QUALITY_FILTERS = True
USE_GLOBAL_FEATURES = True
SUBTRACT_GLOBAL_BASELINE = True

MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Train/validation split
VAL_FRAC = 0.2
SPLIT_SEED = 0

# Depth-2 model
NUM_LAYERS = 2
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

# Embedding clustering
N_CLUSTERS = 9
EMBEDDING_VARIANT = "local"
RESIDUALIZE_GLOBAL_FOR_CLUSTERING = False
CLUSTERING_METHOD = "gmm"
COVARIANCE_TYPE = "full"
STANDARDIZE_EMBEDDINGS = True
PCA_DIM = 32
CLUSTER_SEED = 0

# Strictly non-overlapping patch sampling
PATCH_RADIUS = 2
PATCH_PACKING_RESTARTS = 20
PATCH_SEED = 0

# Accuracy filtering
ACCURATE_PATCH_FRACTION = 0.50
ACCURATE_PATCH_MIN_PER_CLUSTER = 1

# Output
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = (
    f"depth{NUM_LAYERS}_clusters{N_CLUSTERS}_radius{PATCH_RADIUS}_"
    f"accurate{int(100 * ACCURATE_PATCH_FRACTION)}_{RUN_TIMESTAMP}"
)
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / RUN_NAME
FIGURES_DIR = SAVE_DIR / "figures"
TABLES_DIR = SAVE_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)


**Helpers**

In [ ]:
def save_figure(fig, name, *, dpi=300):
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{name}.{suffix}", dpi=dpi, bbox_inches="tight")


def select_target_column(values, target_index=0):
    arr = np.asarray(values)
    if arr.ndim == 1:
        return arr
    return arr[:, int(target_index)]


def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, (torch.dtype, np.dtype)):
        return str(value)
    return value


def split_flat_by_graph(values, graphs):
    values = np.asarray(values)
    output = []
    offset = 0
    for graph in graphs:
        n_nodes = int(graph.x.shape[0])
        output.append(values[offset:offset + n_nodes].copy())
        offset += n_nodes
    if offset != len(values):
        raise ValueError(f"Consumed {offset} values but received {len(values)}.")
    return output


## 1. Load And Filter Data

In [ ]:
from src.data.filters import (
    filter_graphs_by_blacklist,
    filter_graphs_by_marker_diversity,
    filter_graphs_by_metadata,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
    load_graph_blacklist_from_dir,
)
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    add_log_metadata_features,
    attach_metadata_to_graphs,
    fill_missing_metadata_for_group,
    infer_global_dim,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    promote_metadata_to_graph_tensors,
    snapshot_graph_metadata,
    strip_graph_metadata,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
metadata = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, metadata, exclude_keys=None)
graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    marker_names = [f"marker_{i}" for i in range(int(graphs[0].x.shape[1]))]
marker_names = list(marker_names)
print(f"Loaded {len(graphs)} organoids and {len(marker_names)} markers.")


In [ ]:
if EXCLUDE_TIMEPOINT:
    graphs = filter_graphs_by_metadata(
        graphs,
        key="timepoint",
        drop_values={EXCLUDE_TIMEPOINT},
        missing="keep",
        inplace=False,
        print_summary=True,
    )

if FILTER_BLACKLISTED_ORGANOIDS:
    blacklist = load_graph_blacklist_from_dir(data_dir)
    graphs = filter_graphs_by_blacklist(graphs, blacklist, print_summary=True)
else:
    blacklist = set()

if APPLY_QUALITY_FILTERS:
    graphs = fill_missing_metadata_for_group(
        graphs,
        field="complexity",
        fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
        dataset=MISSING_COMPLEXITY_GROUP["dataset"],
        timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
    )
    graphs, spherical = filter_graphs_by_sphericity(
        graphs,
        max_sphericity=SPHERICITY_MAX,
        print_summary=True,
        return_rejected=True,
    )
    _ = filter_graphs_by_marker_diversity(
        spherical,
        min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
        print_summary=True,
    )
    graphs = filter_graphs_by_numeric_metadata(
        graphs,
        key="complexity",
        min_value=COMPLEXITY_MIN,
        allow_missing=False,
        inplace=False,
        print_summary=True,
    )

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, target_outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    target_outlier_info = None

print(f"After filtering: {len(graphs)} organoids.")


## 2. Split And Train A Depth-2 Model

In [ ]:
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.target_transforms import (
    AsinhStandardizeTransform,
    ChainedTargetTransform,
    GlobalBaselineResidualTransform,
    standardize_graph_global_features,
)
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term

FIELD_SPECS = [{
    "meta_keys": ["log_surface_area", "log_volume", "log_volume_over_area", "log_num_cells"],
    "attr_name": "global_feat",
    "kind": "graph_vector",
    "dtype": torch.float32,
}]

if USE_GLOBAL_FEATURES:
    graphs = promote_metadata_to_graph_tensors(
        add_log_metadata_features(graphs, inplace=False),
        FIELD_SPECS,
        inplace=False,
    )

g_train_raw, g_val_raw, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    key_fn=graph_metadata_key,
    inplace=False,
)
g_train = strip_graph_metadata(copy.deepcopy(g_train_raw), inplace=False)
g_val = strip_graph_metadata(copy.deepcopy(g_val_raw), inplace=False)
val_metadata_lookup = snapshot_graph_metadata(g_val_raw)

if USE_GLOBAL_FEATURES:
    standardize_graph_global_features(
        g_train,
        g_val,
        attr_name="global_feat",
        robust=False,
    )

if SUBTRACT_GLOBAL_BASELINE:
    target_transform = ChainedTargetTransform([
        GlobalBaselineResidualTransform(
            hidden_dim=HIDDEN_DIM,
            dropout=DROPOUT,
            lr=LR,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            num_workers=NUM_WORKERS,
        ),
        AsinhStandardizeTransform(robust=True),
    ])
else:
    target_transform = AsinhStandardizeTransform(robust=True)

target_transform.fit(g_train)
target_transform.transform_graphs(g_train, in_place=True)
target_transform.transform_graphs(g_val, in_place=True)
print(f"Split -> train: {len(g_train)} | validation: {len(g_val)}")


In [ ]:
model = GINCurvature(
    n_markers=int(g_train[0].x.shape[1]),
    global_dim=infer_global_dim(g_train),
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    residual=RESIDUAL,
    norm=NORM,
)
train_config = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=[WeightedLossTerm(
        name="edge",
        fn=edge_loss_term,
        weight=EDGE_LOSS_WEIGHT,
        params=EDGE_LOSS_PARAMS,
    )],
)
model, metrics, history = train(model, g_train, g_val, train_config)
print(metrics)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="train")
ax.plot(history["val_loss"], label="validation")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Depth-2 model training")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "training_loss")
plt.show()


## 3. Cluster Validation Node Embeddings

In [ ]:
from src.analysis.motif_clustering import run_embedding_clustering

extraction, clustering_result, cluster_summary_raw = run_embedding_clustering(
    graphs=g_val,
    model=model,
    batch_size=BATCH_SIZE,
    center_only=False,
    embedding_variant=EMBEDDING_VARIANT,
    residualize_global=RESIDUALIZE_GLOBAL_FOR_CLUSTERING,
    clustering=CLUSTERING_METHOD,
    n_clusters=N_CLUSTERS,
    covariance_type=COVARIANCE_TYPE,
    standardize=STANDARDIZE_EMBEDDINGS,
    pca_dim=PCA_DIM,
    seed=CLUSTER_SEED,
    marker_names=marker_names,
)

y_true, y_pred, log_var = target_transform.inverse_distribution(
    extraction.y_true,
    extraction.y_pred,
    log_var=extraction.log_var,
    graphs=g_val,
)
y_true = select_target_column(y_true, TARGET_INDEX_FOR_ANALYSIS)
y_pred = select_target_column(y_pred, TARGET_INDEX_FOR_ANALYSIS)
labels = np.asarray(clustering_result.labels, dtype=int)

# Relabel clusters by median true curvature for stable, interpretable display order.
unique_labels = np.sort(np.unique(labels))
cluster_medians = {
    int(cluster): float(np.nanmedian(y_true[labels == cluster]))
    for cluster in unique_labels
}
old_to_new = {
    old: new
    for new, old in enumerate(sorted(unique_labels, key=lambda x: cluster_medians[int(x)]))
}
labels = np.asarray([old_to_new[int(label)] for label in labels], dtype=int)
clustering_result.labels = labels

labels_by_graph = split_flat_by_graph(labels, g_val_raw)
y_true_by_graph = split_flat_by_graph(y_true, g_val_raw)
y_pred_by_graph = split_flat_by_graph(y_pred, g_val_raw)

cluster_summary_df = pd.DataFrame([{
    "cluster": int(cluster),
    "n_validation_nodes": int(np.sum(labels == cluster)),
    "median_true_curvature": float(np.nanmedian(y_true[labels == cluster])),
    "mean_true_curvature": float(np.nanmean(y_true[labels == cluster])),
    "mean_predicted_curvature": float(np.nanmean(y_pred[labels == cluster])),
    "median_absolute_error": float(
        np.nanmedian(np.abs(y_pred[labels == cluster] - y_true[labels == cluster]))
    ),
} for cluster in np.sort(np.unique(labels))])
display(cluster_summary_df)


## 4. Sample Strictly Non-Overlapping Radius-2 Patches

In [ ]:
from scripts.export_patch_composition_tables import (
    build_adjacency,
    graph_identity,
    nodes_within_hops,
    original_node_id,
)


def sample_non_overlapping_patches(
    graph,
    *,
    radius,
    seed,
    n_restarts,
):
    adjacency = build_adjacency(graph)
    candidates = [
        np.asarray(nodes_within_hops(adjacency, center, radius), dtype=np.int64)
        for center in range(int(graph.x.shape[0]))
    ]
    if not candidates:
        return []

    rng = np.random.default_rng(seed)
    orders = [
        np.asarray(sorted(range(len(candidates)), key=lambda i: (len(candidates[i]), i)))
    ]
    orders.extend(rng.permutation(len(candidates)) for _ in range(int(n_restarts)))

    best = []
    for order in orders:
        used_nodes = set()
        selected = []
        for center in order:
            patch_nodes = candidates[int(center)]
            if any(int(node) in used_nodes for node in patch_nodes):
                continue
            selected.append((int(center), patch_nodes))
            used_nodes.update(int(node) for node in patch_nodes)
        if len(selected) > len(best):
            best = selected
    return sorted(best, key=lambda item: item[0])


def verify_patch_disjointness(patches):
    used = set()
    for center, nodes in patches:
        node_set = set(int(node) for node in nodes)
        if used.intersection(node_set):
            return False
        used.update(node_set)
    return True


In [ ]:
patch_rows = []
organoid_sampling_rows = []

for graph_idx, graph in enumerate(g_val_raw):
    patches = sample_non_overlapping_patches(
        graph,
        radius=PATCH_RADIUS,
        seed=PATCH_SEED + graph_idx,
        n_restarts=PATCH_PACKING_RESTARTS,
    )
    if not verify_patch_disjointness(patches):
        raise RuntimeError(f"Patch overlap detected in validation graph {graph_idx}.")

    x = graph.x.detach().cpu().numpy()
    identity = graph_identity(graph)
    n_sampled_cells = sum(len(nodes) for _, nodes in patches)
    organoid_sampling_rows.append({
        "graph_index": graph_idx,
        **identity,
        "n_organoid_cells": int(graph.x.shape[0]),
        "n_patches": len(patches),
        "n_cells_in_sampled_patches": n_sampled_cells,
        "sampled_cell_fraction": (
            n_sampled_cells / int(graph.x.shape[0])
            if int(graph.x.shape[0]) > 0
            else np.nan
        ),
        "patches_are_node_disjoint": True,
    })

    for patch_index, (center, patch_nodes) in enumerate(patches):
        patch_x = x[patch_nodes]
        row = {
            "graph_index": graph_idx,
            **identity,
            "patch_index": patch_index,
            "center_node_id": center,
            "center_node_original_id": original_node_id(graph, center),
            "patch_radius": PATCH_RADIUS,
            "patch_node_ids": json.dumps([int(node) for node in patch_nodes]),
            "n_cells_patch": len(patch_nodes),
            "cluster": int(labels_by_graph[graph_idx][center]),
            "center_true_curvature": float(y_true_by_graph[graph_idx][center]),
            "center_predicted_curvature": float(y_pred_by_graph[graph_idx][center]),
        }
        row["center_absolute_error"] = abs(
            row["center_predicted_curvature"] - row["center_true_curvature"]
        )
        for marker_idx, marker in enumerate(marker_names):
            positive = patch_x[:, marker_idx] > 0.5
            row[f"n_marker_{marker_idx}_positive"] = int(positive.sum())
            row[f"frac_marker_{marker_idx}_positive"] = float(positive.mean())
            row[f"center_marker_{marker_idx}_positive"] = int(
                x[center, marker_idx] > 0.5
            )
        patch_rows.append(row)

patch_df = pd.DataFrame(patch_rows)
organoid_sampling_df = pd.DataFrame(organoid_sampling_rows)
print(f"Sampled {len(patch_df):,} non-overlapping patches.")
display(organoid_sampling_df.describe(include="all"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(organoid_sampling_df["n_patches"], bins=20, color="#4C78A8")
axes[0].set_xlabel("non-overlapping patches per organoid")
axes[0].set_ylabel("organoids")
axes[0].set_title("Patch count")

axes[1].hist(patch_df["n_cells_patch"], bins=20, color="#59A14F")
axes[1].set_xlabel("cells per radius-2 patch")
axes[1].set_ylabel("patches")
axes[1].set_title("Patch size")
fig.tight_layout()
save_figure(fig, "patch_sampling_distributions")
plt.show()


## 5. Marker Statistics By Center-Cell Cluster

In [ ]:
def patch_marker_long_table(patches, stage):
    rows = []
    for marker_idx, marker in enumerate(marker_names):
        columns = [
            "graph_index",
            "organoid_str",
            "patch_index",
            "cluster",
            "center_absolute_error",
            "n_cells_patch",
            f"n_marker_{marker_idx}_positive",
            f"frac_marker_{marker_idx}_positive",
        ]
        marker_df = patches[columns].copy()
        marker_df = marker_df.rename(columns={
            f"n_marker_{marker_idx}_positive": "n_positive",
            f"frac_marker_{marker_idx}_positive": "fraction_positive",
        })
        marker_df["marker_index"] = marker_idx
        marker_df["marker"] = marker
        marker_df["stage"] = stage
        rows.append(marker_df)
    return pd.concat(rows, ignore_index=True)


def summarize_patch_markers(long_df):
    summary = (
        long_df.groupby(["stage", "cluster", "marker_index", "marker"], as_index=False)
        .agg(
            n_patches=("fraction_positive", "size"),
            mean_fraction_positive=("fraction_positive", "mean"),
            std_fraction_positive=("fraction_positive", "std"),
            median_fraction_positive=("fraction_positive", "median"),
            q25_fraction_positive=("fraction_positive", lambda x: x.quantile(0.25)),
            q75_fraction_positive=("fraction_positive", lambda x: x.quantile(0.75)),
            mean_positive_cells=("n_positive", "mean"),
            mean_patch_size=("n_cells_patch", "mean"),
        )
    )
    summary["std_fraction_positive"] = summary["std_fraction_positive"].fillna(0.0)
    summary["sem_fraction_positive"] = (
        summary["std_fraction_positive"]
        / np.sqrt(summary["n_patches"].clip(lower=1))
    )
    return summary


patch_marker_all_df = patch_marker_long_table(patch_df, "all_patches")
marker_summary_all_df = summarize_patch_markers(patch_marker_all_df)
display(marker_summary_all_df.head(12))


In [ ]:
def plot_marker_fraction_heatmaps(
    all_summary,
    accurate_summary,
    *,
    filename,
):
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
    stages = [
        (all_summary, "All non-overlapping patches"),
        (accurate_summary, "Best-predicted 50% within each cluster"),
    ]
    image = None
    for ax, (summary_df, title) in zip(axes, stages):
        table = (
            summary_df.pivot(
                index="cluster",
                columns="marker",
                values="mean_fraction_positive",
            )
            .reindex(columns=marker_names)
            .sort_index()
        )
        sem_table = (
            summary_df.pivot(
                index="cluster",
                columns="marker",
                values="sem_fraction_positive",
            )
            .reindex(index=table.index, columns=table.columns)
        )
        image = ax.imshow(table.to_numpy(float), aspect="auto", vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(np.arange(len(marker_names)))
        ax.set_xticklabels(marker_names, rotation=45, ha="right")
        ax.set_yticks(np.arange(len(table.index)))
        ax.set_yticklabels([f"C{cluster}" for cluster in table.index])
        ax.set_xlabel("marker")
        ax.set_title(title)
        for row in range(table.shape[0]):
            for col in range(table.shape[1]):
                mean = table.iloc[row, col]
                sem = sem_table.iloc[row, col]
                if np.isfinite(mean):
                    ax.text(
                        col,
                        row,
                        f"{mean:.2f}\n±{sem:.2f}",
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="white" if mean < 0.35 else "black",
                    )
    axes[0].set_ylabel("center-cell embedding cluster")
    fig.colorbar(image, ax=axes, label="mean patch fraction positive", shrink=0.85)
    fig.suptitle("Patch marker composition by center-cell embedding cluster")
    fig.subplots_adjust(left=0.08, right=0.91, bottom=0.18, top=0.88, wspace=0.18)
    save_figure(fig, filename)
    plt.show()
    return fig, axes


def plot_patch_marker_distributions(long_df, title, filename):
    n_markers = len(marker_names)
    n_cols = min(4, n_markers)
    n_rows = int(np.ceil(n_markers / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.0 * n_cols, 3.1 * n_rows),
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()
    clusters = np.sort(long_df["cluster"].unique())
    for ax, marker in zip(axes, marker_names):
        marker_df = long_df[long_df["marker"] == marker]
        values = [
            marker_df.loc[marker_df["cluster"] == cluster, "fraction_positive"].to_numpy(float)
            for cluster in clusters
        ]
        boxplot = ax.boxplot(values, showfliers=False, patch_artist=True)
        for box in boxplot["boxes"]:
            box.set_facecolor("#4C78A8")
        ax.set_title(marker)
        ax.set_xticks(np.arange(1, len(clusters) + 1))
        ax.set_xticklabels([f"C{cluster}" for cluster in clusters], rotation=45)
        ax.set_ylim(-0.03, 1.03)
        ax.grid(axis="y", alpha=0.2)
    for ax in axes[n_markers:]:
        ax.set_visible(False)
    for row in range(n_rows):
        axes[row * n_cols].set_ylabel("patch fraction positive")
    fig.suptitle(title)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_figure(fig, filename)
    plt.show()
    return fig, axes


## 6. Keep The Best-Predicted 50% Within Each Cluster

In [ ]:
def best_patch_mask_by_cluster(
    patches,
    *,
    fraction,
    min_per_cluster=1,
):
    if not (0 < fraction <= 1):
        raise ValueError("fraction must lie in (0, 1].")
    keep = np.zeros(len(patches), dtype=bool)
    rows = []
    for cluster in np.sort(patches["cluster"].unique()):
        positions = np.flatnonzero(patches["cluster"].to_numpy() == cluster)
        errors = patches.iloc[positions]["center_absolute_error"].to_numpy(float)
        finite_positions = positions[np.isfinite(errors)]
        n_available = len(finite_positions)
        n_keep = min(
            n_available,
            max(int(min_per_cluster), int(np.ceil(fraction * n_available))),
        )
        if n_keep:
            ranked = finite_positions[
                np.argsort(
                    patches.iloc[finite_positions]["center_absolute_error"].to_numpy(float),
                    kind="mergesort",
                )
            ]
            chosen = ranked[:n_keep]
            keep[chosen] = True
            threshold = float(
                patches.iloc[chosen]["center_absolute_error"].max()
            )
        else:
            threshold = np.nan
        rows.append({
            "cluster": int(cluster),
            "n_available_patches": n_available,
            "n_kept_patches": n_keep,
            "kept_fraction": n_keep / n_available if n_available else np.nan,
            "absolute_error_threshold": threshold,
            "median_absolute_error_all": float(np.nanmedian(errors)),
            "median_absolute_error_kept": (
                float(np.nanmedian(patches.loc[keep & (patches["cluster"] == cluster), "center_absolute_error"]))
                if n_keep
                else np.nan
            ),
        })
    return keep, pd.DataFrame(rows)


accurate_patch_mask, accuracy_filter_summary_df = best_patch_mask_by_cluster(
    patch_df,
    fraction=ACCURATE_PATCH_FRACTION,
    min_per_cluster=ACCURATE_PATCH_MIN_PER_CLUSTER,
)
accurate_patch_df = patch_df.loc[accurate_patch_mask].copy()
patch_marker_accurate_df = patch_marker_long_table(
    accurate_patch_df,
    "best_predicted_half",
)
marker_summary_accurate_df = summarize_patch_markers(patch_marker_accurate_df)

print(
    f"Retained {len(accurate_patch_df):,}/{len(patch_df):,} patches "
    f"({len(accurate_patch_df) / len(patch_df):.1%})."
)
display(accuracy_filter_summary_df)


In [ ]:
plot_marker_fraction_heatmaps(
    marker_summary_all_df,
    marker_summary_accurate_df,
    filename="patch_marker_fraction_heatmaps_all_vs_accurate",
)
plot_patch_marker_distributions(
    patch_marker_all_df,
    "Patch marker fractions | all non-overlapping patches",
    "patch_marker_distributions_all",
)
plot_patch_marker_distributions(
    patch_marker_accurate_df,
    "Patch marker fractions | best-predicted 50% within each cluster",
    "patch_marker_distributions_accurate_half",
)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
clusters = accuracy_filter_summary_df["cluster"].to_numpy(int)
width = 0.38
ax.bar(
    clusters - width / 2,
    accuracy_filter_summary_df["n_available_patches"],
    width=width,
    label="all patches",
    color="#B8C4CE",
)
ax.bar(
    clusters + width / 2,
    accuracy_filter_summary_df["n_kept_patches"],
    width=width,
    label="best-predicted half",
    color="#4C78A8",
)
ax.set_xticks(clusters)
ax.set_xticklabels([f"C{cluster}" for cluster in clusters])
ax.set_xlabel("center-cell embedding cluster")
ax.set_ylabel("sampled patches")
ax.set_title("Patch counts before and after within-cluster accuracy filtering")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "patch_counts_by_cluster")
plt.show()


## 7. Save Tables And Settings

In [ ]:
patch_df.to_csv(TABLES_DIR / "non_overlapping_patch_table.csv", index=False)
accurate_patch_df.to_csv(TABLES_DIR / "accurate_half_patch_table.csv", index=False)
organoid_sampling_df.to_csv(TABLES_DIR / "organoid_patch_sampling_summary.csv", index=False)
patch_marker_all_df.to_csv(TABLES_DIR / "patch_marker_values_all.csv", index=False)
patch_marker_accurate_df.to_csv(TABLES_DIR / "patch_marker_values_accurate_half.csv", index=False)
marker_summary_all_df.to_csv(TABLES_DIR / "patch_marker_summary_all.csv", index=False)
marker_summary_accurate_df.to_csv(
    TABLES_DIR / "patch_marker_summary_accurate_half.csv",
    index=False,
)
accuracy_filter_summary_df.to_csv(
    TABLES_DIR / "accuracy_filter_summary_by_cluster.csv",
    index=False,
)
cluster_summary_df.to_csv(TABLES_DIR / "validation_cluster_summary.csv", index=False)


In [ ]:
settings = {
    "experiment_group": EXPERIMENT_GROUP,
    "dataset_name": DATASET_NAME,
    "target_indices": TARGET_INDICES,
    "target_index_for_analysis": TARGET_INDEX_FOR_ANALYSIS,
    "filtering": {
        "exclude_timepoint": EXCLUDE_TIMEPOINT,
        "filter_blacklisted_organoids": FILTER_BLACKLISTED_ORGANOIDS,
        "blacklist_n_keys": len(blacklist),
        "apply_quality_filters": APPLY_QUALITY_FILTERS,
        "missing_complexity_group": MISSING_COMPLEXITY_GROUP,
        "sphericity_max": SPHERICITY_MAX,
        "spherical_marker_diversity_min": SPHERICAL_MARKER_DIVERSITY_MIN,
        "complexity_min": COMPLEXITY_MIN,
        "interpolate_target_outliers": INTERPOLATE_TARGET_OUTLIERS,
        "outlier_clip_quantiles": OUTLIER_CLIP_QUANTILES,
    },
    "split": {
        "validation_fraction": VAL_FRAC,
        "seed": SPLIT_SEED,
        "n_train_graphs": len(g_train),
        "n_validation_graphs": len(g_val),
    },
    "model": {
        "class": "GINCurvature",
        "num_layers": NUM_LAYERS,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "norm": NORM,
        "residual": RESIDUAL,
        "use_global_features": USE_GLOBAL_FEATURES,
        "subtract_global_baseline": SUBTRACT_GLOBAL_BASELINE,
    },
    "training": {
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "edge_loss_weight": EDGE_LOSS_WEIGHT,
        "edge_loss_params": EDGE_LOSS_PARAMS,
    },
    "clustering": {
        "n_clusters": N_CLUSTERS,
        "embedding_variant": EMBEDDING_VARIANT,
        "residualize_global": RESIDUALIZE_GLOBAL_FOR_CLUSTERING,
        "method": CLUSTERING_METHOD,
        "covariance_type": COVARIANCE_TYPE,
        "standardize_embeddings": STANDARDIZE_EMBEDDINGS,
        "pca_dim": PCA_DIM,
        "seed": CLUSTER_SEED,
        "labels_reordered_by": "median true curvature",
    },
    "patch_sampling": {
        "radius": PATCH_RADIUS,
        "strict_node_disjointness": True,
        "packing_restarts": PATCH_PACKING_RESTARTS,
        "seed": PATCH_SEED,
        "cluster_assignment": "cluster of center cell",
        "marker_statistic_observation_unit": "patch",
    },
    "accuracy_filter": {
        "metric": "absolute center-cell prediction error",
        "fraction_kept_per_cluster": ACCURATE_PATCH_FRACTION,
        "minimum_kept_per_cluster": ACCURATE_PATCH_MIN_PER_CLUSTER,
    },
    "marker_names": marker_names,
    "n_graphs_after_filtering": len(graphs),
    "n_sampled_patches": len(patch_df),
    "n_accurate_patches": len(accurate_patch_df),
}

with open(SAVE_DIR / "settings.json", "w") as handle:
    json.dump(jsonable(settings), handle, indent=2)
with open(SAVE_DIR / "results.pkl", "wb") as handle:
    pickle.dump({
        "metrics": metrics,
        "history": history,
        "split_info": split_info,
        "target_outlier_info": target_outlier_info,
        "cluster_labels": labels,
        "cluster_label_mapping": old_to_new,
        "patch_df": patch_df,
        "accurate_patch_mask": accurate_patch_mask,
        "marker_summary_all_df": marker_summary_all_df,
        "marker_summary_accurate_df": marker_summary_accurate_df,
    }, handle)

print("Saved patch sampling analysis to", SAVE_DIR)
